# Relay BYODT 框架

BYODT(Bring Your Own Datatype) 框架允许用户在 TVM 中注册和使用自定义数据类型。

In [1]:
import set_env

In [2]:
import numpy as np
import pytest

import tvm
import tvm.topi.testing
import tvm.testing
from tvm import relay
from tvm.relay.testing.layers import batch_norm_infer
from tvm.target.datatype import (
    create_lower_func,  # 创建自定义数据类型的Lower函数
    create_min_lower_func,  # 创建最小值Lower函数
    lower_call_pure_extern,  # 处理外部函数调用
    lower_ite,  # 处理条件分支
    register,  # 注册自定义数据类型
    register_min_func,  # 注册最小值函数
    register_op,  # 注册操作
)
from tvm.tir.op import call_pure_extern
from tvm.script import tir as T

## 设置 `myfloat` 自定义数据类型

注册名为 `myfloat` 的自定义数据类型（底层是 `float`），并注册相应的算子函数，使 TVM 能够正确处理这种自定义数据类型。

要使用外部库中的数据类型算子，应首先加载包含该数据类型实现的库：
```
CDLL("libposit.so", RTLD_GLOBAL)
```
本示例中，使用的数据类型库已内置在 TVM 中，因此无需显式加载任何库：

In [3]:
def setup_myfloat():
    # 可以为自定义数据类型选择任意代码，只要大于128且未被其他数据类型使用
    register("myfloat", 131)  # 注册名为myfloat的数据类型，代码为131
    # 注册float到myfloat的转换算子
    register_op(
        create_lower_func({(32, 32): "FloatToCustom32"}), "Cast", "llvm", "float", "myfloat"
    )
    # 注册myfloat到float的转换算子
    register_op(
        create_lower_func({(32, 32): "Custom32ToFloat"}), "Cast", "llvm", "myfloat", "float"
    )
    # 注册myfloat类型的加法算子
    register_op(create_lower_func({32: "Custom32Add"}), "Add", "llvm", "myfloat")
    # 注册myfloat类型的减法算子
    register_op(
        create_lower_func(
            {
                32: "Custom32Sub",
            }
        ),
        "Sub",
        "llvm",
        "myfloat",
    )
    # 注册myfloat类型的乘法算子
    register_op(create_lower_func({32: "Custom32Mul"}), "Mul", "llvm", "myfloat")
    # 注册myfloat类型的浮点常量处理
    register_op(
        create_lower_func(
            {
                32: "FloatToCustom32",
            }
        ),
        "FloatImm",
        "llvm",
        "myfloat",
    )
    # 注册myfloat类型的除法算子
    register_op(
        create_lower_func(
            {
                32: "Custom32Div",
            }
        ),
        "Div",
        "llvm",
        "myfloat",
    )
    # 注册myfloat类型的最大值算子
    register_op(create_lower_func({32: "Custom32Max"}), "Max", "llvm", "myfloat")
    # 注册myfloat类型的平方根内联函数
    register_op(
        create_lower_func({32: "Custom32Sqrt"}),
        "Call",
        "llvm",
        "myfloat",
        intrinsic_name="tir.sqrt",
    )
    # 注册myfloat类型的指数内联函数
    register_op(
        create_lower_func({32: "Custom32Exp"}),
        "Call",
        "llvm",
        "myfloat",
        intrinsic_name="tir.exp",
    )
    # 注册myfloat类型的对数内联函数
    register_op(
        create_lower_func({32: "Custom32Log"}),
        "Call",
        "llvm",
        "myfloat",
        intrinsic_name="tir.log",
    )
    # 注册myfloat类型的sigmoid内联函数
    register_op(
        create_lower_func({32: "Custom32Sigmoid"}),
        "Call",
        "llvm",
        "myfloat",
        intrinsic_name="tir.sigmoid",
    )
    # 注册myfloat类型的tanh内联函数
    register_op(
        create_lower_func({32: "Custom32Tanh"}),
        "Call",
        "llvm",
        "myfloat",
        intrinsic_name="tir.tanh",
    )
    # 注册myfloat类型的条件分支内联函数
    register_op(lower_ite, "Call", "llvm", "myfloat", intrinsic_name="tir.if_then_else")
    # 注册myfloat类型的外部函数调用内联函数
    register_op(
        lower_call_pure_extern, "Call", "llvm", "myfloat", intrinsic_name="tir.call_pure_extern"
    )

    # 注册myfloat类型的最小值函数
    register_min_func(create_min_lower_func({32: "MinCustom32"}, "myfloat"), "myfloat")

In [4]:
def convert_ndarray(dst_dtype, array):
    """
    将NumPy数组转换为指定的数据类型
    
    参数:
        dst_dtype: 目标数据类型
        array: 输入的NumPy数组
    
    返回:
        转换为目标数据类型后的数组
    """
    # 创建一个与输入数组形状和数据类型相同的Relay变量
    x = relay.var("x", shape=array.shape, dtype=str(array.dtype))
    # 创建一个将输入变量转换为目标数据类型的函数
    cast = relay.Function([x], x.astype(dst_dtype))
    # 在禁用向量化优化的上下文中执行转换（自定义数据类型可能尚未实现向量化操作）
    with tvm.transform.PassContext(config={"tir.disable_vectorize": True}):
        # 使用图形执行器评估转换函数并应用于输入数组
        return relay.create_executor("graph").evaluate(cast)(array)
        
def change_dtype(src, dst, module, params):
    """将模块中的常量和函数从源数据类型转换为目标数据类型
    
    参数:
        src: 源数据类型，如 'float32'
        dst: 目标数据类型，如自定义的 'myfloat'
        module: TVM Relay IR 模块
        params: 模型参数字典，键为参数名，值为参数值
        
    返回:
        tuple: (转换数据类型后的模块, 转换数据类型后的参数字典)
    """
    # 使用 Relay 前端的 ChangeDatatype 转换器将模块中的数据类型从 src 转换为 dst
    module = relay.frontend.ChangeDatatype(src, dst)(module)
    # 使用类型推断变换确保所有类型注解都正确更新
    module = relay.transform.InferType()(module)
    # 遍历参数字典，将每个参数值转换为目标数据类型
    params = {k: convert_ndarray(dst, v) for k, v in params.items()}
    # 返回转换后的模块和参数
    return module, params


def compare(module, input, src_dtype, dst_dtype, rtol, atol, params={}, target="llvm"):
    """比较两种不同数据类型下模型执行结果的一致性
    
    该函数首先使用源数据类型执行模型获取基准结果，然后将模型转换为目标数据类型并执行，
    最后将目标数据类型的执行结果转回源数据类型并与基准结果进行比较，确保误差在可接受范围内。
    
    参数:
        module: TVM Relay IR 模块
        input: 模型输入数据列表
        src_dtype: 源数据类型，如 'float32'
        dst_dtype: 目标数据类型，如自定义的 'myfloat'
        rtol: 相对误差容限
        atol: 绝对误差容限
        params: 模型参数字典，默认为空字典
        target: 目标执行平台，默认为 'llvm'
    
    异常:
        AssertionError: 如果两种数据类型的执行结果差异超过容限，会触发断言失败
    """
    # 应用类型推断变换确保模块中的类型注解正确
    module = relay.transform.InferType()(module)
    # 简化模块中的推断操作，优化计算图
    module = relay.transform.SimplifyInference()(module)

    # 使用源数据类型执行模型，获取基准结果（作为"正确"结果）
    correct = relay.create_executor("graph", mod=module).evaluate()(*input, **params)
    # 将模块和参数从源数据类型转换为目标数据类型
    module, converted_params = change_dtype(src_dtype, dst_dtype, module, params)
    # 将所有输入数据转换为目标数据类型
    x_converted = [convert_ndarray(dst_dtype, arr) for arr in input]

    # 自定义数据类型尚未实现向量化操作，需要禁用向量化优化
    with tvm.transform.PassContext(config={"tir.disable_vectorize": True}):
        # 使用转换后的模块和参数在目标平台上执行模型
        maybe_correct = relay.create_executor("graph", mod=module, target=target).evaluate()(
            *x_converted, **converted_params
        )
        # 注意：当前此函数仅支持比较单输出模型
        # 将目标数据类型的执行结果转回源数据类型，以便比较
        maybe_correct_converted = convert_ndarray(src_dtype, maybe_correct)
    # 断言两种执行结果的差异在允许的容限范围内
    np.testing.assert_allclose(
        maybe_correct_converted.numpy(), correct.numpy(), rtol=rtol, atol=atol
    )


def run_ops(src_dtype, dst_dtype, rtol=1e-7, atol=1e-7):
    """使用两种不同的数据类型运行相同的操作，用于比较自定义数据类型与原始数据类型的计算结果
    
    参数:
        src_dtype: 源数据类型，通常为标准数据类型如float32
        dst_dtype: 目标数据类型，通常为自定义数据类型如myfloat
        rtol: 相对误差容限
        atol: 绝对误差容限
    """
    # 用于一元操作和二元操作的第一个输入的形状
    shape1 = (5, 10, 5)
    # 二元操作的第二个输入的形状
    shape2 = (5,)

    def check_unary_op(op, src_dtype, dst_dtype, shape):
        """检查一元操作在不同数据类型下的执行结果
        
        参数:
            op: 要测试的操作函数
            src_dtype: 源数据类型
            dst_dtype: 目标数据类型
            shape: 输入张量的形状
        """
        t1 = relay.TensorType(shape, src_dtype)
        x = relay.var("x", t1)
        z = op(x)  # 应用操作到输入变量
        x_data = np.random.rand(*shape).astype(t1.dtype)  # 生成随机测试数据

        # 创建IR模块
        module = tvm.IRModule.from_expr(relay.Function([x], z))

        # 比较两种数据类型的执行结果
        compare(module, (x_data,), src_dtype, dst_dtype, rtol, atol)

    # 测试各种一元操作
    for op in [
        relay.nn.softmax,      # 计算softmax激活函数
        tvm.relay.log,         # 计算自然对数
        tvm.relay.exp,         # 计算指数函数
        tvm.relay.sqrt,        # 计算平方根
        tvm.relay.rsqrt,       # 计算平方根的倒数
        tvm.relay.sigmoid,     # 计算sigmoid激活函数
        tvm.relay.tanh,        # 计算双曲正切激活函数
        relay.nn.relu,         # 计算ReLU激活函数
        relay.nn.batch_flatten,# 对输入进行批展平操作
    ]:
        check_unary_op(op, src_dtype, dst_dtype, shape1)

    # 测试在4维数据上的一元操作
    for op in [relay.nn.max_pool2d, relay.nn.avg_pool2d, relay.nn.global_avg_pool2d]:
        shape_2d = (3, 32, 32, 32)  # 4维张量形状: (批大小, 通道数, 高度, 宽度)
        check_unary_op(op, src_dtype, dst_dtype, shape_2d)

    def check_binary_op(opfunc, src_dtype, dst_dtype):
        """检查二元操作在不同数据类型下的执行结果
        
        参数:
            opfunc: 要测试的二元操作函数
            src_dtype: 源数据类型
            dst_dtype: 目标数据类型
        """
        t1 = relay.TensorType(shape1, src_dtype)
        t2 = relay.TensorType(shape2, src_dtype)
        x = relay.var("x", t1)
        y = relay.var("y", t2)
        z = opfunc(x, y)  # 应用二元操作到两个输入变量
        x_data = np.random.rand(*shape1).astype(t1.dtype)  # 生成第一个输入的随机数据
        y_data = np.random.rand(*shape2).astype(t2.dtype)  # 生成第二个输入的随机数据
        module = tvm.IRModule.from_expr(relay.Function([x, y], z))

        # 比较两种数据类型的执行结果
        compare(module, (x_data, y_data), src_dtype, dst_dtype, rtol, atol)

    # 测试各种二元操作
    for op in [
        relay.add,       # 加法操作
        relay.subtract,  # 减法操作
        relay.divide,    # 除法操作
        relay.multiply,  # 乘法操作
    ]:
        check_binary_op(op, src_dtype, dst_dtype)

    # 注意：我们原本想测试tvm_if_then_else
    # 但Relay.IfNode不会被转换为这个内在函数，
    # 所以为了保持测试与relay的一致性，我们决定不进行单元测试
    # 注：tvm_if_then_else在mobile_net模型中会被测试


def run_model(get_workload, input, src_dtype, dst_dtype, rtol=1e-4, atol=1e-4):
    """运行完整模型并比较不同数据类型的执行结果
    
    参数:
        get_workload: 一个函数，返回模型模块和参数
        input: 模型的输入数据
        src_dtype: 源数据类型
        dst_dtype: 目标数据类型
        rtol: 相对误差容限
        atol: 绝对误差容限
    """
    module, params = get_workload()  # 获取模型结构和参数

    # 这里不生成随机数据
    # 因为那样会导致输出数据的值都在相似的范围内
    compare(module, input, src_dtype, dst_dtype, rtol, atol, params)


def run_conv2d(src_dtype, dst_dtype, rtol=1e-7, atol=1e-4):
    """专门测试卷积操作在不同数据类型下的表现
    
    参数:
        src_dtype: 源数据类型
        dst_dtype: 目标数据类型
        rtol: 相对误差容限
        atol: 绝对误差容限
    """
    def run_test_conv2d(
        src_dtype,
        dst_dtype,
        scale,  # 随机数据的范围缩放因子
        dshape, # 输入数据形状
        kshape, # 卷积核形状
        padding=(1, 1),  # 填充大小
        groups=1,        # 分组卷积的组数
        dilation=(1, 1), # 膨胀率
        **attrs,         # 其他卷积属性
    ):
        """运行特定配置的卷积测试
        
        参数:
            scale: 随机数据的范围缩放因子
            dshape: 输入数据形状
            kshape: 卷积核形状
            padding: 填充大小
            groups: 分组卷积的组数
            dilation: 膨胀率
            **attrs: 其他卷积属性
        """
        x = relay.var("x", shape=dshape, dtype=src_dtype)  # 定义输入变量
        w = relay.var("w", shape=kshape, dtype=src_dtype)  # 定义卷积核变量
        # 创建卷积操作
        y = relay.nn.conv2d(x, w, padding=padding, dilation=dilation, groups=groups, **attrs)
        # 创建IR模块
        module = tvm.IRModule.from_expr(relay.Function([x, w], y))
        # 生成随机测试数据
        data = np.random.uniform(-scale, scale, size=dshape).astype(src_dtype)
        kernel = np.random.uniform(-scale, scale, size=kshape).astype(src_dtype)

        # 比较两种数据类型的执行结果
        compare(module, (data, kernel), src_dtype, dst_dtype, rtol, atol)

    # 测试深度可分离卷积
    dshape = (1, 32, 18, 18)  # 输入形状: (批大小, 通道数, 高度, 宽度)
    kshape = (32, 1, 3, 3)    # 卷积核形状: (输出通道数, 输入通道数/组数, 高度, 宽度)
    run_test_conv2d(
        src_dtype,
        dst_dtype,
        1,  # 缩放因子为1
        dshape,
        kshape,
        padding=(1, 1),
        channels=32,     # 输出通道数
        groups=32,       # 组数等于通道数，实现深度可分离卷积
        kernel_size=(3, 3),  # 卷积核大小
    )

    # 注意：CUDA对于'direct'调度被禁用:
    # https://github.com/dmlc/tvm/pull/3070#issuecomment-486597553
    # 测试分组卷积
    dshape = (1, 32, 18, 18)
    kshape = (32, 4, 3, 3)  # 32=8*4，8是组数，4是每组的通道数
    run_test_conv2d(
        src_dtype,
        dst_dtype,
        1,
        dshape,
        kshape,
        padding=(1, 1),
        channels=32,     # 输出通道数
        groups=8,        # 组数为8
        kernel_size=(3, 3),
    )
    # 另一个分组卷积测试
    dshape = (1, 32, 18, 18)
    kshape = (64, 1, 3, 3)  # 64=32*2，32是组数
    run_test_conv2d(
        src_dtype,
        dst_dtype,
        1,
        dshape,
        kshape,
        padding=(1, 1),
        channels=64,     # 输出通道数
        groups=32,       # 组数为32
        kernel_size=(3, 3),
    )

    # 测试普通卷积
    dshape = (1, 3, 224, 224)  # 标准图像输入大小
    kshape = (10, 3, 3, 3)     # 10个3x3的卷积核，每个卷积核有3个输入通道
    run_test_conv2d(
        src_dtype, dst_dtype, 1, dshape, kshape, padding=(1, 1), channels=10, kernel_size=(3, 3)
    )

    # 测试膨胀卷积（空洞卷积）
    dshape = (1, 3, 18, 18)
    kshape = (10, 3, 3, 3)
    run_test_conv2d(
        src_dtype,
        dst_dtype,
        1,
        dshape,
        kshape,
        padding=(1, 1),
        channels=10,
        kernel_size=(3, 3),
        dilation=(3, 3),  # 设置膨胀率为3
    )

def run_batchnorm(src_dtype, dst_dtype, rtol=1e-6, atol=1e-6):
    """
    测试批归一化（Batch Normalization）操作在不同数据类型下的一致性
    
    参数:
        src_dtype: 源数据类型，通常是标准数据类型
        dst_dtype: 目标数据类型，通常是自定义数据类型
        rtol: 相对误差容限，默认为1e-6
        atol: 绝对误差容限，默认为1e-6
    """
    # 设置输入数据形状 (通道数, 高度, 宽度)
    shape = (3, 32, 32)
    # 创建Relay张量类型
    t = relay.TensorType(shape, src_dtype)
    # 创建输入变量
    x = relay.var("x", t)
    # 构建批归一化推理层（不使用缩放参数）
    bn = batch_norm_infer(data=x, epsilon=2e-5, scale=False, name="bn_x")
    # 创建函数，捕获所有自由变量
    f = relay.Function(relay.analysis.free_vars(bn), bn)

    # 生成随机输入数据
    x_data = np.random.rand(*shape).astype(t.dtype)
    # 从表达式创建IR模块
    module = tvm.IRModule.from_expr(f)

    # 创建零值数组作为批归一化的参数（均值、方差、缩放和偏移）
    zero_data = np.zeros((32), "float32")
    # 比较源数据类型和目标数据类型下的批归一化结果
    compare(
        module,
        (x_data, zero_data, zero_data, zero_data, zero_data),  # 输入数据和批归一化参数
        src_dtype,
        dst_dtype,
        rtol,
        atol,
    )



In [5]:
setup_myfloat()
run_ops("float32", "custom[myfloat]32", rtol=1e-6, atol=1e-6)
run_conv2d("float32", "custom[myfloat]32", rtol=1e-6, atol=1e-6)
run_batchnorm("float32", "custom[myfloat]32", rtol=1e-6, atol=1e-6)

## 测试 TVM 自定义数据类型（myfloat）的TIR（Tensor IR）降低（Lowering）过程

In [7]:
class TestMyfloatLowering(tvm.testing.CompareBeforeAfter):
    """
    测试TVM自定义数据类型（myfloat）的TIR（Tensor IR）降低（Lowering）过程
    
    该测试类继承自CompareBeforeAfter，用于验证自定义数据类型操作是否正确地转换为底层实现
    通过比较转换前后的IR代码，确保myfloat类型的加法操作被正确地降低为外部函数调用
    """
    # 在类加载时注册myfloat自定义数据类型
    # setup_myfloat()

    # 设置要测试的转换：降低自定义数据类型
    transform = tvm.tir.transform.LowerCustomDatatypes()

    def before(self):
        """
        定义转换前的原始TIR函数，使用myfloat自定义数据类型
        
        返回:
            使用myfloat类型的原始TIR函数
        """
        # 定义自定义数据类型标识符
        dtype = "custom[myfloat]32"

        @T.prim_func
        def func(A_data: T.handle(dtype)):
            # 设置目标为LLVM
            T.func_attr({"target": T.target("llvm")})
            # 从原始句柄创建myfloat类型的缓冲区
            A = T.Buffer(16, dtype=dtype, data=A_data)
            # 分配myfloat类型的输出缓冲区
            B_data = T.allocate([16], dtype=dtype)
            B = T.Buffer(16, dtype=dtype, data=B_data)
            # 执行简单的元素级加法操作（每个元素加1.0）
            for i in range(16):
                B[i] = A[i] + 1.0

        return func

    def expected(self):
        """
        定义转换后的期望TIR函数，验证自定义数据类型的降低实现
        
        返回:
            降低后的期望TIR函数
        """
        # 定义自定义数据类型标识符
        dtype = "custom[myfloat]32"

        @T.prim_func
        def func(A_data: T.handle(dtype)):
            # 设置目标为LLVM
            T.func_attr({"target": T.target("llvm")})
            # 将myfloat类型的输入缓冲区视为uint32（底层表示）
            A_uint32 = T.Buffer(16, "uint32", data=A_data)
            # 分配uint32类型的输出缓冲区
            B_data = T.allocate([16], dtype="uint32")
            B_uint32 = T.Buffer(16, "uint32", data=B_data)
            # 执行降低后的操作，通过外部函数调用实现自定义数据类型的加法
            for i in range(16):
                B_uint32[i] = T.call_pure_extern(
                    "uint32",
                    "FloatToCustom32",  # 将浮点数转换回myfloat格式的外部函数
                    T.call_pure_extern("float32", "Custom32ToFloat", A_uint32[i]) + T.float32(1),  # 将myfloat转换为浮点数并执行加法
                )

        return func
